In [6]:
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML

# Load datasets
orders = pd.read_csv("../Source data/OListDatasets/olist_orders_dataset.csv")
customers = pd.read_csv("../Source data/OListDatasets/olist_customers_dataset.csv")

# Convert purchase timestamp
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Merge customers
orders = orders.merge(customers[['customer_id','customer_unique_id']], on='customer_id', how='left')

# Extract order month
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

# Determine cohort month (first purchase month per customer)
cohort = orders.groupby('customer_unique_id')['order_month'].min().reset_index()
cohort.columns = ['customer_unique_id','cohort_month']

# Merge back
orders = orders.merge(cohort, on='customer_unique_id')

# Calculate months since first purchase
orders['months_since'] = (orders['order_month'] - orders['cohort_month']).apply(lambda x: x.n)

# Cohort counts
cohort_data = orders.groupby(['cohort_month','months_since'])['customer_unique_id'].nunique().reset_index()

# Pivot to cohort table
cohort_pivot = cohort_data.pivot(index='cohort_month', columns='months_since', values='customer_unique_id')

# Normalize to retention rates
cohort_size = cohort_pivot[0]
cohort_retention = cohort_pivot.divide(cohort_size, axis=0).round(3)

# Convert Periods to string for Plotly
cohort_retention.index = cohort_retention.index.astype(str)

# --- Show cohort table ---
print("Cohort Analysis Table (fraction of customers retained):")
display(cohort_retention)

# --- Percentages version ---
cohort_retention_pct = (cohort_retention * 100).round(1)
print("\nCohort Analysis Table (% of customers retained):")
display(cohort_retention_pct)

# --- Heatmap visualization ---
fig = px.imshow(
    cohort_retention_pct,
    labels=dict(x="Months Since First Purchase", y="Cohort Month", color="Retention (%)"),
    x=cohort_retention_pct.columns,
    y=cohort_retention_pct.index,
    aspect="auto",
    color_continuous_scale="Viridis",
    title="Customer Cohort Analysis"
)

fig.update_layout(height=600, template="plotly_white")

# Proper inline display
display(HTML(fig.to_html(include_plotlyjs='cdn')))

Cohort Analysis Table (fraction of customers retained):


months_since,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,1.0,NaN,NaN,NaN,NaN,NaN,0.003,NaN,NaN,0.003,NaN,0.003,NaN,0.003,NaN,0.003,NaN,0.003,0.006,0.006
2016-12,1.0,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,1.0,0.004,0.003,0.001,0.004,0.001,0.005,0.001,0.001,NaN,0.004,0.001,0.008,0.004,0.001,0.001,0.003,0.004,0.001,NaN
2017-02,1.0,0.002,0.003,0.001,0.004,0.001,0.002,0.002,0.002,0.002,0.001,0.003,0.002,0.002,0.001,0.001,0.001,0.002,NaN,NaN
2017-03,1.0,0.005,0.004,0.004,0.003,0.002,0.002,0.003,0.003,0.001,0.004,0.002,0.002,0.001,0.002,0.002,0.001,0.002,NaN,NaN
2017-04,1.0,0.006,0.002,0.002,0.003,0.003,0.003,0.003,0.003,0.002,0.003,0.001,0.001,0.000,0.001,0.001,0.002,NaN,NaN,NaN
2017-05,1.0,0.005,0.005,0.004,0.003,0.003,0.004,0.002,0.003,0.003,0.003,0.003,0.003,0.000,0.002,0.003,NaN,NaN,NaN,NaN
2017-06,1.0,0.005,0.004,0.004,0.003,0.004,0.004,0.002,0.001,0.002,0.003,0.004,0.002,0.001,0.002,NaN,NaN,NaN,NaN,NaN



Cohort Analysis Table (% of customers retained):


months_since,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,100.0,NaN,NaN,NaN,NaN,NaN,0.3,NaN,NaN,0.3,NaN,0.3,NaN,0.3,NaN,0.3,NaN,0.3,0.6,0.6
2016-12,100.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,100.0,0.4,0.3,0.1,0.4,0.1,0.5,0.1,0.1,NaN,0.4,0.1,0.8,0.4,0.1,0.1,0.3,0.4,0.1,NaN
2017-02,100.0,0.2,0.3,0.1,0.4,0.1,0.2,0.2,0.2,0.2,0.1,0.3,0.2,0.2,0.1,0.1,0.1,0.2,NaN,NaN
2017-03,100.0,0.5,0.4,0.4,0.3,0.2,0.2,0.3,0.3,0.1,0.4,0.2,0.2,0.1,0.2,0.2,0.1,0.2,NaN,NaN
2017-04,100.0,0.6,0.2,0.2,0.3,0.3,0.3,0.3,0.3,0.2,0.3,0.1,0.1,0.0,0.1,0.1,0.2,NaN,NaN,NaN
2017-05,100.0,0.5,0.5,0.4,0.3,0.3,0.4,0.2,0.3,0.3,0.3,0.3,0.3,0.0,0.2,0.3,NaN,NaN,NaN,NaN
2017-06,100.0,0.5,0.4,0.4,0.3,0.4,0.4,0.2,0.1,0.2,0.3,0.4,0.2,0.1,0.2,NaN,NaN,NaN,NaN,NaN
